In [25]:
library("R.matlab")
library("tidyverse")
library("afex")
library("BayesFactor")

In [26]:
extract_metrics <- function(filepath, group) {
  mat <- readMat(filepath)
  if (is.null(mat$meanRT) || is.null(mat$meanMT) ||
      length(mat$meanRT) < 3 || length(mat$meanMT) < 3) {
    message(paste("Skipping file due to missing or invalid data:", filepath))
    return(NULL)
  }
  data.frame(
    Subject = basename(filepath),
    Group = group,
    Block = c('baseline', 'early_learning', 'late_learning'),
    ResponseTime = as.numeric(mat$meanRT[1:3]),
    MovementTime = as.numeric(mat$meanMT[1:3])
  )
}

adult_files <- list.files('adult_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)
child_files <- list.files('children_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)

data_adult <- map_dfr(adult_files, ~extract_metrics(.x, 'adult'))
data_child <- map_dfr(child_files, ~extract_metrics(.x, 'child'))

data_all <- bind_rows(data_adult, data_child)

glimpse(data_all)

Rows: 72
Columns: 5
$ Subject      <chr> "VML_MEG_011_Final_Results.mat", "VML_MEG_011_Final_Resul…
$ Group        <chr> "adult", "adult", "adult", "adult", "adult", "adult", "ad…
$ Block        <chr> "baseline", "early_learning", "late_learning", "baseline"…
$ ResponseTime <dbl> 0.3590000, 0.3308000, 0.3260000, 0.3590000, 0.3308000, 0.…
$ MovementTime <dbl> 1.0180000, 1.1364667, 1.0935333, 1.0180000, 1.1364667, 1.…


In [27]:
anova_rt <- aov_ez(
  id = "Subject",
  dv = "ResponseTime",
  data = data_all,
  between = "Group",
  within = "Block"
)
print("2-way mixed ANOVA for Response Time:")
print(anova_rt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group

Contrasts set to contr.sum for the following variables: Group



[1] "2-way mixed ANOVA for Response Time:"
Anova Table (Type 3 tests)

Response: ResponseTime
       Effect          df  MSE       F   ges p.value
1       Group       1, 22 0.03 8.95 **  .270    .007
2       Block 1.62, 35.58 0.00    0.73  .003    .461
3 Group:Block 1.62, 35.58 0.00    0.03 <.001    .941
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 
Anova Table (Type 3 tests)

Response: ResponseTime
       Effect          df  MSE       F   ges p.value
1       Group       1, 22 0.03 8.95 **  .270    .007
2       Block 1.62, 35.58 0.00    0.73  .003    .461
3 Group:Block 1.62, 35.58 0.00    0.03 <.001    .941
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [28]:
anova_mt <- aov_ez(
  id = "Subject",
  dv = "MovementTime",
  data = data_all,
  between = "Group",
  within = "Block"
)
print("2-way mixed ANOVA for Movement Time:")
print(anova_mt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group

Contrasts set to contr.sum for the following variables: Group



[1] "2-way mixed ANOVA for Movement Time:"
Anova Table (Type 3 tests)

Response: MovementTime
       Effect          df  MSE         F  ges p.value
1       Group       1, 22 0.02      1.05 .033    .316
2       Block 1.33, 29.34 0.01 24.08 *** .239   <.001
3 Group:Block 1.33, 29.34 0.01      2.34 .030    .129
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 
Anova Table (Type 3 tests)

Response: MovementTime
       Effect          df  MSE         F  ges p.value
1       Group       1, 22 0.02      1.05 .033    .316
2       Block 1.33, 29.34 0.01 24.08 *** .239   <.001
3 Group:Block 1.33, 29.34 0.01      2.34 .030    .129
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [29]:
data_all$Subject <- as.factor(data_all$Subject)
data_all$Group <- as.factor(data_all$Group)
data_all$Block <- as.factor(data_all$Block)

bf_rt <- anovaBF(
  ResponseTime ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)
print("Bayes Factor for Response Time:")
print(bf_rt)

bf_mt <- anovaBF(
  MovementTime ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)
print("Bayes Factor for Movement Time:")
print(bf_mt)

[1] "Bayes Factor for Response Time:"
Bayes factor analysis
--------------
[1] Group + Subject                       : 5.092738  ±3.57%
[2] Block + Subject                       : 0.2164803 ±1.08%
[3] Group + Block + Subject               : 1.06013   ±3.9%
[4] Group + Block + Group:Block + Subject : 0.2222818 ±7.85%

Against denominator:
  ResponseTime ~ Subject 
---
Bayes factor type: BFlinearModel, JZS

[1] "Bayes Factor for Movement Time:"
Bayes factor analysis
--------------
[1] Group + Subject                       : 0.5209596 ±1.44%
[2] Block + Subject                       : 163705.5  ±1.1%
[3] Group + Block + Subject               : 98744.28  ±1.21%
[4] Group + Block + Group:Block + Subject : 86390.21  ±1.47%

Against denominator:
  MovementTime ~ Subject 
---
Bayes factor type: BFlinearModel, JZS



In [30]:
mat_test <- readMat(adult_files[1])
str(mat_test)
print(names(mat_test))

List of 15
 $ AA       : num [1:130, 1] -20.67 -17.64 1.97 -5.87 -40.16 ...
 $ ACC      : num [1:130, 1] 1 1 1 1 1 1 1 1 1 1 ...
 $ DIRECTION: num [1:130, 1] 1 -1 1 1 -1 -1 1 1 -1 -1 ...
 $ DV       : num [1:130, 1] 3.16 4.55 4.78 5.05 4.59 ...
 $ EA       : num [1:130, 1] -16.09 -8.14 -5.89 -5.52 -18.87 ...
 $ IDE      : num [1:130, 1] -16.09 -8.13 -5.89 -5.49 -18.98 ...
 $ KEEP     : num [1:130, 1] 1 1 1 1 1 1 1 1 1 1 ...
 $ MT       : num [1:130, 1] 0.955 0.889 0.923 0.957 0.992 ...
 $ OUTLIER  : int [1:130, 1] 0 0 0 0 0 0 0 0 0 0 ...
 $ PA       : num [1:130, 1] 2616 1420 521 1164 998 ...
 $ PV       : num [1:130, 1] 36.5 49.4 45.8 56.1 48.3 ...
 $ RT       : num [1:130, 1] 0.505 0.448 0.39 0.394 0.321 0.283 0.317 0.355 0.262 0.315 ...
 $ TV       : num [1:130, 1] 0.612 0.572 0.546 0.522 0.466 0.504 0.482 0.484 0.426 0.421 ...
 $ meanMT   : num [1, 1:5] 1.02 1.14 1.09 1.07 1.1
 $ meanRT   : num [1, 1:5] 0.359 0.331 0.326 0.315 0.339
 - attr(*, "header")=List of 3
  ..$ description: